In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(r"D:\Data Analyst\Project\SQL\Netflix\data\netflix_titles.csv")

In [4]:
df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,NaN,NaN,NaN,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


In [5]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [7]:
df['date_added_dt'] = pd.to_datetime(df['date_added'], errors='coerce')


In [8]:
print(f"Successfully converted: {df['date_added_dt'].notna().sum()} rows")

Successfully converted: 8709 rows


In [9]:
# 1. Parse date_added
df['year_added']  = df['date_added_dt'].dt.year.astype('Int64')
df['month_added'] = df['date_added_dt'].dt.month.astype('Int64')
df['month_name']  = df['date_added_dt'].dt.strftime('%B')

In [10]:
# 2. Extract numeric duration
df['duration_int'] = df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_unit'] = df['duration'].str.extract(r'([A-Za-z]+)')


In [11]:
# 3. Fill nulls
df['director'].fillna('Unknown', inplace=True)
df['cast'].fillna('Unknown', inplace=True)
df['country'].fillna('Unknown', inplace=True)
df['rating'].fillna('Not Rated', inplace=True)

In [12]:
df.isnull().sum()

show_id           0
type              0
title             0
director          0
cast              0
country           0
date_added       10
release_year      0
rating            0
duration          3
listed_in         0
description       0
date_added_dt    98
year_added       98
month_added      98
month_name       98
duration_int      3
duration_unit     3
dtype: int64

In [13]:
# 4. Exploded versions for multi-value columns 
df_genres = df[['show_id','title','type']].copy()
df_genres['genre'] = df['listed_in'].str.split(', ')
df_genres = df_genres.explode('genre')

In [14]:
df_genres


,show_id,title,type,genre
0,s1,Dick Johnson Is Dead,Movie,Documentaries
1,s2,Blood & Water,TV Show,International TV Shows
1,s2,Blood & Water,TV Show,TV Dramas
1,s2,Blood & Water,TV Show,TV Mysteries
2,s3,Ganglands,TV Show,Crime TV Shows
...,...,...,...,...
8805,s8806,Zoom,Movie,Children & Family Movies
8805,s8806,Zoom,Movie,Comedies
8806,s8807,Zubaan,Movie,Dramas
8806,s8807,Zubaan,Movie,International Movies


In [15]:
# 1. Create the copy
df_countries = df[['show_id','title','type']].copy()

# 2. Split the string into a list
df_countries['country'] = df['country'].str.split(',')

# 3. Explode into separate rows
df_countries = df_countries.explode('country')

# 4. NOW strip the spaces from the specific column
df_countries['country'] = df_countries['country'].str.strip()

In [16]:
df_countries

,show_id,title,type,country
0,s1,Dick Johnson Is Dead,Movie,United States
1,s2,Blood & Water,TV Show,South Africa
2,s3,Ganglands,TV Show,Unknown
3,s4,Jailbirds New Orleans,TV Show,Unknown
4,s5,Kota Factory,TV Show,India
...,...,...,...,...
8802,s8803,Zodiac,Movie,United States
8803,s8804,Zombie Dumb,TV Show,Unknown
8804,s8805,Zombieland,Movie,United States
8805,s8806,Zoom,Movie,United States


In [17]:
df_cast = df[['show_id','title','type']].copy()
df_cast['actor'] = df['cast'].str.split(', ')
df_cast = df_cast.explode('actor')


In [18]:
df_cast

,show_id,title,type,actor
0,s1,Dick Johnson Is Dead,Movie,Unknown
1,s2,Blood & Water,TV Show,Ama Qamata
1,s2,Blood & Water,TV Show,Khosi Ngema
1,s2,Blood & Water,TV Show,Gail Mabalane
1,s2,Blood & Water,TV Show,Thabang Molaba
...,...,...,...,...
8806,s8807,Zubaan,Movie,Manish Chaudhary
8806,s8807,Zubaan,Movie,Meghna Malik
8806,s8807,Zubaan,Movie,Malkeet Rauni
8806,s8807,Zubaan,Movie,Anita Shabdish


In [19]:
# 5. Export clean files
df.to_csv('netflix_clean.csv', index=False)
df_genres.to_csv('netflix_genres.csv', index=False)
df_countries.to_csv('netflix_countries.csv', index=False)
df_cast.to_csv('netflix_cast.csv', index=False)

In [20]:
print(df.shape, df.dtypes)

(8807, 18) show_id                  object
type                     object
title                    object
director                 object
cast                     object
country                  object
date_added               object
release_year              int64
rating                   object
duration                 object
listed_in                object
description              object
date_added_dt    datetime64[ns]
year_added                Int64
month_added               Int64
month_name               object
duration_int            float64
duration_unit            object
dtype: object
